In [9]:
!pip install numpy-financial
from pathlib import Path

import numpy as np
import numpy_financial as npf
import pandas as pd

In [10]:
"""
Base fictícia para portfólio financeiro (v2).
TODOS OS DADOS SÃO SINTÉTICOS. Nenhuma empresa ou pessoa real é representada.
"""

# ------------ Parâmetros ------------
SEED = 42
rng = np.random.default_rng(SEED)
OUT = Path("data")
OUT.mkdir(exist_ok=True)

DATA_REF = pd.Timestamp("2026-10-31")
MESES_HIST = pd.date_range("2024-11-01", "2026-10-01", freq="MS")
MESES_GER = pd.date_range("2024-08-01", "2026-10-01", freq="MS")
MESES_PROJ = pd.date_range("2026-11-01", periods=12, freq="MS")

SALDO_INICIAL = 1_500_000.0
DESP_FIXA_BASE = 150_000.0
ALIQUOTA_IMPOSTOS = 0.08
WACC = 0.14                                   # a.a., ilustrativo
CAPEX = 2_000_000.0                           # pago no 1º mês da projeção

ATRASO = {"pontual": (-2, 4), "atraso_leve": (12, 8), "atraso_grave": (55, 25)}

CENARIOS = {
    # atraso_extra: meses adicionais na curva de recebimento (carteira pagando mais tarde)
    "Pessimista": dict(cresc=-0.005, inad=0.06,  cv=0.62, g_fixas=0.008, atraso_extra=1),
    "Base":       dict(cresc=0.008,  inad=0.03,  cv=0.58, g_fixas=0.004, atraso_extra=0),
    "Otimista":   dict(cresc=0.018,  inad=0.015, cv=0.55, g_fixas=0.003, atraso_extra=0),
}

FLUXOS_PROJETO = {                            # t0 = -CAPEX; t1..t5 = fluxos anuais
    "Pessimista": [-CAPEX, 350_000, 450_000, 550_000, 600_000, 650_000],
    "Base":       [-CAPEX, 600_000, 750_000, 850_000, 900_000, 950_000],
    "Otimista":   [-CAPEX, 800_000, 1_000_000, 1_200_000, 1_300_000, 1_400_000],
}

PDD = {"A vencer": 0.00, "1-30": 0.01, "31-60": 0.05,
       "61-90": 0.15, "91-180": 0.40, ">180": 0.80}


# ------------ Clientes ------------
def gerar_clientes() -> pd.DataFrame:
    perfis = np.array(["pontual"] * 24 + ["atraso_leve"] * 11 + ["atraso_grave"] * 5)
    rng.shuffle(perfis)
    n = len(perfis)
    return pd.DataFrame({
        "cliente_id": [f"C{i:03d}" for i in range(1, n + 1)],
        "nome": [f"Cliente Fictício {i:03d}" for i in range(1, n + 1)],
        "segmento": rng.choice(["Varejo", "Indústria", "Serviços", "Agronegócio", "Saúde"], n),
        "prazo_dias": rng.choice([30, 45, 60], n, p=[0.40, 0.35, 0.25]),
        "perfil_pagamento": perfis,
        "dados_ficticios": True,
    })


# ------------ Faturas ------------
def gerar_faturas(clientes: pd.DataFrame) -> pd.DataFrame:
    linhas, seq = [], 1
    for c in clientes.itertuples():
        mu, sd = ATRASO[c.perfil_pagamento]
        ticket = rng.lognormal(np.log(22_000), 0.5)
        for mes in MESES_GER:
            if rng.random() > 0.85:
                continue
            emissao = mes + pd.Timedelta(days=int(rng.integers(0, 28)))
            venc = emissao + pd.Timedelta(days=int(c.prazo_dias))
            pgto = venc + pd.Timedelta(days=int(round(rng.normal(mu, sd))))
            calote = c.perfil_pagamento == "atraso_grave" and rng.random() < 0.10
            if calote or pgto > DATA_REF:
                pgto = pd.NaT
            linhas.append({
                "fatura_id": f"F{seq:06d}",
                "cliente_id": c.cliente_id,
                "data_emissao": emissao,
                "data_vencimento": venc,
                "data_pagamento": pgto,
                "valor": round(ticket * rng.uniform(0.7, 1.3), 2),
            })
            seq += 1
    df = pd.DataFrame(linhas)
    df["status"] = np.where(df["data_pagamento"].isna(), "Em aberto", "Pago")
    df["dados_ficticios"] = True
    return df


# ------------ Histórico ------------
def gerar_historico(faturas: pd.DataFrame) -> pd.DataFrame:
    fat = (faturas.set_index("data_emissao")["valor"]
           .resample("MS").sum().reindex(MESES_HIST, fill_value=0))
    rec = (faturas.dropna(subset=["data_pagamento"]).set_index("data_pagamento")["valor"]
           .resample("MS").sum().reindex(MESES_HIST, fill_value=0))
    return pd.DataFrame({"mes": MESES_HIST, "faturamento": fat.values,
                         "recebimentos": rec.values, "dados_ficticios": True})


# ------------ Curva de recebimento ------------
def curva_recebimento(faturas: pd.DataFrame, max_meses: int = 4) -> pd.Series:
    """% do valor recebido k meses após a emissão (k = dias // 30, truncado em max_meses).
    Usa só faturas emitidas há mais de 6 meses, para reduzir o viés de censura
    (faturas recentes ainda não tiveram tempo de ser pagas)."""
    corte = DATA_REF - pd.Timedelta(days=180)
    base = faturas[faturas["data_pagamento"].notna() & (faturas["data_emissao"] <= corte)].copy()
    base["k"] = ((base["data_pagamento"] - base["data_emissao"]).dt.days // 30).clip(0, max_meses)
    curva = base.groupby("k")["valor"].sum()
    return curva / curva.sum()


def receber(serie: list, curva: pd.Series, inad: float, atraso_extra: int = 0) -> float:
    """Recebimento do mês corrente: serie[-1] é o faturamento do mês corrente."""
    return (1 - inad) * sum(p * serie[-1 - int(k) - atraso_extra] for k, p in curva.items())


# ------------ Aging ------------
def gerar_aging(faturas: pd.DataFrame):
    ab = faturas[faturas["data_pagamento"].isna()].copy()
    ab["dias_atraso"] = (DATA_REF - ab["data_vencimento"]).dt.days
    ab["faixa"] = pd.cut(ab["dias_atraso"],
                         bins=[-np.inf, 0, 30, 60, 90, 180, np.inf],
                         labels=list(PDD))
    ab["pdd_estimada"] = ab["valor"] * ab["faixa"].astype(str).map(PDD)
    resumo = (ab.groupby("faixa", observed=False)
                .agg(qtd_faturas=("fatura_id", "count"),
                     saldo=("valor", "sum"),
                     pdd_estimada=("pdd_estimada", "sum"))
                .reset_index())
    resumo["pct_carteira"] = resumo["saldo"] / resumo["saldo"].sum()
    return ab, resumo


# ------------ Fluxo projetado ------------
def projetar_caixa(hist: pd.DataFrame, curva: pd.Series) -> pd.DataFrame:
    fat_base = hist["faturamento"].iloc[-3:].mean()
    linhas = []
    for nome, p in CENARIOS.items():
        serie = list(hist["faturamento"])
        saldo_com, saldo_sem = SALDO_INICIAL, SALDO_INICIAL
        entrada_projeto = FLUXOS_PROJETO[nome][1] / 12   # fluxo do ano 1, diluído por mês
        for i, mes in enumerate(MESES_PROJ, start=1):
            fat = fat_base * (1 + p["cresc"]) ** i
            serie.append(fat)
            receb = receber(serie, curva, p["inad"], p["atraso_extra"])
            cv = fat * p["cv"]
            imp = fat * ALIQUOTA_IMPOSTOS
            fixas = DESP_FIXA_BASE * (1 + p["g_fixas"]) ** i
            operacional = receb - cv - imp - fixas
            capex = CAPEX if i == 1 else 0.0
            saldo_sem += operacional
            saldo_com += operacional - capex + entrada_projeto
            linhas.append({
                "cenario": nome, "mes": mes, "faturamento": fat,
                "recebimentos": receb, "custos_variaveis": cv, "impostos": imp,
                "despesas_fixas": fixas, "fluxo_operacional": operacional,
                "capex": capex, "entrada_projeto": entrada_projeto,
                "saldo_sem_projeto": saldo_sem, "saldo_caixa": saldo_com,
                "dados_ficticios": True,
            })
    return pd.DataFrame(linhas)


def resumo_liquidez(proj: pd.DataFrame) -> pd.DataFrame:
    linhas = []
    for nome, g in proj.groupby("cenario", sort=False):
        g = g.reset_index(drop=True)
        i_min = g["saldo_caixa"].idxmin()
        depois = g.loc[i_min:]
        rec = depois.loc[depois["saldo_caixa"] >= 0, "mes"]
        linhas.append({
            "cenario": nome,
            "caixa_minimo": g.loc[i_min, "saldo_caixa"],
            "mes_do_minimo": g.loc[i_min, "mes"],
            "mes_recuperacao": rec.iloc[0] if len(rec) else pd.NaT,
            "saldo_final_12m": g["saldo_caixa"].iloc[-1],
            "necessidade_financiamento": max(0.0, -g["saldo_caixa"].min()),
            "dados_ficticios": True,
        })
    return pd.DataFrame(linhas)


# ------------ Viabilidade ------------
def payback(fluxos, taxa: float = 0.0) -> float:
    acum = 0.0
    for t, f in enumerate(fluxos):
        valor = f / (1 + taxa) ** t
        anterior, acum = acum, acum + valor
        if t > 0 and acum >= 0:
            return t - 1 + (-anterior / valor)
    return np.nan


def indicadores_projeto() -> pd.DataFrame:
    return pd.DataFrame([{
        "cenario": nome,
        "investimento": -f[0],
        "wacc": WACC,
        "vpl": npf.npv(WACC, f),
        "tir": npf.irr(f),
        "payback_simples_anos": payback(f),
        "payback_descontado_anos": payback(f, WACC),
        "dados_ficticios": True,
    } for nome, f in FLUXOS_PROJETO.items()])


# ------------ Execução ------------
if __name__ == "__main__":
    clientes = gerar_clientes()
    faturas = gerar_faturas(clientes)
    historico = gerar_historico(faturas)
    curva = curva_recebimento(faturas)
    aging_det, aging_res = gerar_aging(faturas)
    projecao = projetar_caixa(historico, curva)
    liquidez = resumo_liquidez(projecao)
    projeto = indicadores_projeto()

    # ------------ Verificações de sanidade ------------
    pagas = faturas.dropna(subset=["data_pagamento"])
    assert abs(curva.sum() - 1) < 1e-9, "curva de recebimento não soma 100%"
    assert (faturas["data_vencimento"] > faturas["data_emissao"]).all()
    assert (pagas["data_pagamento"] >= pagas["data_emissao"]).all()
    assert (pagas["data_pagamento"] <= DATA_REF).all()
    assert clientes["perfil_pagamento"].value_counts()["atraso_grave"] == 5
    assert historico["recebimentos"].iloc[0] > 0.5 * historico["recebimentos"].mean(), \
        "cold start ainda presente"

    tabelas = {
        "clientes": clientes, "faturas": faturas, "fluxo_historico": historico,
        "curva_recebimento": curva.rename_axis("meses_apos_emissao").reset_index(name="pct_recebido"),
        "aging_detalhe": aging_det, "aging_resumo": aging_res,
        "fluxo_projetado": projecao, "resumo_liquidez": liquidez,
        "projeto_indicadores": projeto,
    }
    for nome, df in tabelas.items():
        df.to_csv(OUT / f"{nome}.csv", index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(OUT / "portfolio_financeiro.xlsx") as xl:
        for nome, df in tabelas.items():
            df.to_excel(xl, sheet_name=nome[:31], index=False)

    print("Curva de recebimento:\n", curva.round(3).to_string(), "\n")
    print(projeto.round(3).to_string(index=False), "\n")
    print(liquidez.round(0).to_string(index=False), "\n")
    print(aging_res.round(2).to_string(index=False))

Curva de recebimento:
 k
0    0.152
1    0.602
2    0.167
3    0.049
4    0.030 

   cenario  investimento  wacc         vpl   tir  payback_simples_anos  payback_descontado_anos  dados_ficticios
Pessimista     2000000.0  0.14 -282649.936 0.085                 4.077                      NaN             True
      Base     2000000.0  0.14  703414.705 0.267                 2.765                    3.606             True
  Otimista     2000000.0  0.14 1778008.225 0.431                 2.167                    2.653             True 

   cenario  caixa_minimo mes_do_minimo mes_recuperacao  saldo_final_12m  necessidade_financiamento  dados_ficticios
Pessimista     -417247.0    2026-11-01      2027-04-01         485392.0                   417247.0             True
      Base     -310082.0    2026-11-01      2027-01-01        1570698.0                   310082.0             True
  Otimista     -257977.0    2026-11-01      2027-01-01        2399064.0                   257977.0             True 